### Testing RAG Applications 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="500" height="400" style="display: block; margin: auto;">

In [ ]:
# !pip install -qU langchain-chroma

#!pip install -U DeepEval

In [11]:
import deepeval

deepeval.login("confident_us_8k9P7QpyyKgjpa7yzXG0ULlki3JAwq0DPAstgNKA1x0=")

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [12]:
!deepeval set-ollama qwen2.5:latest

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local Ollama model `qwen2.5:latest` for 
all evals that require an LLM.


In [13]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

In [16]:
from deepeval.metrics import AnswerRelevancyMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.tracing import (
    observe,
    update_current_span
    # RetrieverAttributes
)

In [18]:
@observe(type='llm', model='qwen2.5:latest')
def local_llms():
    return ChatOllama(
        base_url="http://localhost:11434",
        model = "qwen2.5:latest",
        temperature=0.5,
        max_tokens = 250
    )
    # return ChatOpenAI(model="gpt-4.1-2025-04-14", max_completion_tokens=300)
    
llm = local_llms()

[Confident AI Trace Log]  Successfully posted trace (0 traces remaining in 
queue, 1 in flight): 
https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/observatory/trac
es/6c1dee24-e08b-4095-a9ce-96ebada72845 
To disable dev logging, set CONFIDENT_TRACE_VERBOSE=0 as an environment 
variable.


In [20]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="llama3.2:latest")
# embedding = OpenAIEmbeddings(model="text-embedding-3-large")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


@observe(metrics=[AnswerRelevancyMetric()])
def retrieve_and_format(question):
    docs = retriever.invoke(question)
    response = format_docs(docs)
    
    update_current_span(
        test_case=LLMTestCase(input=question, actual_output=response)
    )
    
    return response

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


In [21]:
@observe(type="retriever", embedder="text-embedding-3-large")
def retrive_documents(question):
    retrived_context = retrieve_and_format(question)
    
    update_current_span(
        attributes= RetrieverAttributes(
            embedding_input=question,
            retrieval_context= [retrived_context]
        )
    )
    
    return retrived_context



#### Output of the LLM Application

In [22]:

@observe(type="custom", name="RAG Application", metrics=[ContextualRelevancyMetric()])
def rag_application(question):
    actual_response = chain.invoke(question)
    retrived_context = retrive_documents(question)
    
    update_current_span(
        test_case=LLMTestCase(input=question, actual_output=actual_response, retrieval_context=[retrived_context])
    )
    
    return actual_response



### Evaluation of RAG Application

In [ ]:
from deepeval.dataset import Golden
from deepeval import evaluate

goldens = Golden(input="What is MCP")
evaluate(goldens=[goldens], observed_callback=rag_application)


ValueError: You must provide either 'metric_collection' or 'metrics'.